In [1]:
# ============================================================
# [1단계] 내 컴퓨터의 GPU 환경 확인하기
# ============================================================
# AI 학습은 그래픽카드(GPU)를 써야 빨라요.
# 시작 전에 "내 컴퓨터에 적합한 GPU가 있는지", "AI 도구가 GPU를 인식하는지"
# 미리 확인하는 단계예요.
#
# 운동 전에 신발 끈이 잘 묶였는지, 무릎 보호대가 있는지 확인하는 느낌!
# ============================================================

import torch  # PyTorch: AI 학습의 핵심 엔진 (계산을 GPU로 빠르게 해줌)

# PyTorch 버전 출력 (예: 2.8.0+cu128 → CUDA 12.8과 호환되는 버전)
print("torch version:", torch.__version__)

# CUDA 사용 가능 여부 (CUDA = NVIDIA GPU 가속 기술)
# True면 GPU 학습 가능, False면 CPU만 사용 (매우 느림)
print("cuda available:", torch.cuda.is_available())

# 현재 사용 가능한 CUDA 버전
print("cuda version:", torch.version.cuda)

# 0번 GPU(첫 번째 그래픽카드)의 모델명
# 예: "NVIDIA RTX A6000" - 고성능 워크스테이션용 GPU
print("device name:", torch.cuda.get_device_name(0))

# GPU의 compute capability (성능 등급)
# (8, 6) = Ampere 아키텍처, 최신 기능 대부분 지원
print("capability:", torch.cuda.get_device_capability(0))


torch version: 2.8.0+cu128
cuda available: True
cuda version: 12.8
device name: NVIDIA A100-SXM4-80GB
capability: (8, 0)


In [2]:
# ============================================================
# [2단계] 필요한 도구(라이브러리) 한 번에 설치하기
# ============================================================
# VLM(이미지를 이해하는 AI) 학습에 필요한 도구들을 설치해요.
#
# 명령어 앞 '%'는 주피터 노트북 전용 명령
# '!' 와 비슷하지만 더 똑똑하게 처리해요
# '-q' 옵션: quiet(조용히), 설치 중 메시지를 최소화
# ============================================================

# pip(파이썬 패키지 관리자) 자체를 최신 버전으로 업그레이드
# setuptools, wheel: 설치 도구 부속품 (최신 버전이어야 안정적)
%pip install -q --upgrade pip setuptools wheel

# 여러 도구를 한 번에 설치 (역슬래시 '\'는 "다음 줄로 이어진다"는 뜻)
# 버전 표기 ">=4.57.0,<5.0.0" → "4.57.0 이상 5.0.0 미만" 의미
# (특정 버전 범위 안의 것을 설치 → 호환성 문제 방지)
%pip install -q --upgrade \
  "transformers>=4.57.0,<5.0.0" \
  "accelerate>=1.1.0,<2.0.0" \
  "datasets>=3.0.1,<4.0.0" \
  "evaluate>=0.4.3,<1.0.0" \
  "trl>=0.24.0,<1.0.0" \
  "peft>=0.18.0,<1.0.0" \
  "qwen-vl-utils>=0.0.14" \
  "Pillow>=9.4.0" \
  "scikit-learn>=1.3.0" \
  "tensorboard" \
  "wandb" \
  "rich"

# 각 도구가 하는 일:
# - transformers: 허깅페이스의 핵심 AI 모델 라이브러리 (오늘의 주인공)
# - accelerate: 학습 속도를 자동으로 최적화 (멀티GPU 등)
# - datasets: 데이터셋을 쉽게 다운/관리
# - evaluate: AI 성능 평가 도구
# - trl: 강화학습/지도학습용 트레이너 (SFTTrainer 사용 예정)
# - peft: LoRA 같은 효율적 학습 기법 모음
# - qwen-vl-utils: Qwen VL 모델 전용 보조 도구 (이미지 처리)
# - Pillow: 이미지 파일 다루는 도구 (열기, 변환 등)
# - scikit-learn: 머신러닝 도구 (여기선 train/test 데이터 분할용)
# - tensorboard: 학습 과정을 그래프로 시각화
# - wandb: 학습 실험 추적 도구 (Weights & Biases)
# - rich: 터미널 출력을 예쁘게 만드는 도구


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
notebook 7.4.2 requires jupyterlab<4.5,>=4.4.0, but you have jupyterlab 4.5.7 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# [3단계] 허깅페이스 빠른 다운로드 도구 설치
# ============================================================
# hf_transfer: 허깅페이스에서 큰 모델 파일(수 GB)을 다운받을 때
# 일반 방식보다 훨씬 빠르게 받게 해주는 가속기예요.
#
# VLM 모델은 보통 4~10GB라서 다운로드 시간이 많이 걸리는데,
# 이 도구를 설치하면 시간을 크게 줄일 수 있어요.
# ============================================================

%pip install -q hf_transfer


Note: you may need to restart the kernel to use updated packages.


Kernel > Restart Kernel

In [4]:
# ============================================================
# [4단계] 설치한 도구들이 잘 들어갔는지 점검하기
# ============================================================
# 설치 후 커널을 재시작했으니, 도구들을 가져와서(import)
# 각 도구의 버전을 출력해봐요.
# 잘 출력되면 설치 성공! 에러 나면 어디서 문제가 있는지 알 수 있어요.
# ============================================================

import torch              # AI 학습 엔진
import transformers       # AI 모델 라이브러리
import accelerate         # 학습 가속 도구
import datasets           # 데이터셋 관리 도구
import trl                # 트레이너 도구 모음
import peft               # LoRA 등 효율적 학습 도구
import qwen_vl_utils      # Qwen VL 모델 보조 도구

# 하드웨어 정보 (1단계와 동일)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))

# 각 라이브러리 버전 확인
# .__version__: 모듈에 내장된 버전 정보를 보여주는 변수
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)


torch: 2.8.0+cu128
cuda: 12.8
gpu: NVIDIA A100-SXM4-80GB
capability: (8, 0)
transformers: 4.57.6
accelerate: 1.13.0
datasets: 3.6.0
trl: 0.29.1
peft: 0.19.1


In [5]:
# ============================================================
# [5단계] 실제 사용할 모든 도구 한꺼번에 불러오기
# ============================================================
# 4단계가 "설치 확인"이었다면, 여기서는 본격적으로 작업에 쓸 도구들을
# 가져와요. (서랍에서 연필을 책상에 꺼내놓는 느낌)
# ============================================================

# ─── 파이썬 기본 도구 ───
import io       # 입출력 처리 (메모리에서 파일처럼 다루기)
import json     # JSON 데이터 처리 (딕셔너리 ↔ 문자열 변환)
import os       # 운영체제 관련 작업 (파일 경로, 환경변수 등)
import random   # 무작위 숫자 생성

# ─── 데이터 처리 도구 ───
import numpy as np   # 수학/배열 계산 도구 (np는 별명)
import torch         # AI 학습 엔진
import wandb         # 학습 실험 추적 도구

# ─── 이미지 및 데이터셋 ───
from PIL import Image                                   # 이미지 파일 다루기
from datasets import load_dataset                       # 데이터셋 다운로드
from sklearn.model_selection import train_test_split    # 학습/테스트 데이터 분할

# ─── AI 모델 관련 ───
# AutoProcessor: 이미지+텍스트를 AI 입력 형태로 변환하는 종합 처리기
# AutoModelForImageTextToText: 이미지를 보고 텍스트를 생성하는 모델
from transformers import AutoProcessor, AutoModelForImageTextToText

# ─── 학습 도구 ───
# SFTConfig: 지도학습(Supervised Fine-Tuning) 설정
# SFTTrainer: 지도학습 진행 코치 역할
from trl import SFTConfig, SFTTrainer

# ─── Qwen VL 전용 도구 ───
# process_vision_info: 메시지에서 이미지/비디오를 자동 추출
from qwen_vl_utils import process_vision_info

# ─── LoRA(효율적 학습) 설정 ───
from peft import LoraConfig


## 1. 기본 설정

In [6]:
# ============================================================
# [6단계] 기본 설정 - 시드 고정, 모델 이름, 저장 위치 정하기
# ============================================================
# 본격 학습 전에 큰 틀의 설정값들을 정해놓는 단계예요.
# (요리 시작 전에 분량, 시간, 그릇 정하기)
# ============================================================

# wandb(실험 추적 도구) 비활성화
# mode="disabled" → 실험 기록을 외부 서버로 안 보냄 (오프라인 학습)
# 실험 기록을 남기고 싶으면 mode="online"으로 변경
wandb.init(mode="disabled")

# ─── 핵심 설정값 ───

# 랜덤 시드 (재현성 보장 - 매번 같은 결과 나오게)
SEED = 42

# 사용할 AI 모델 이름
# "Qwen/Qwen3-VL-4B-Instruct":
#   - Qwen: 알리바바가 만든 AI 시리즈
#   - Qwen3-VL: 비전(이미지)+언어 모델 (이미지 보고 답할 수 있음)
#   - 4B: 파라미터 약 40억 개 (적당히 큰 사이즈)
#   - Instruct: 명령어/대화에 잘 답하도록 미리 학습된 버전
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# 학습 결과를 저장할 폴더 이름
OUTPUT_DIR = "output_dir_qwen3_vl_4b_instruct_lora"

# ─── 모든 무작위 도구에 같은 시드 설정 ───
# 파이썬 random / NumPy / PyTorch / GPU 모두에 동일한 시드 부여
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 환경 확인 출력 (1단계에서 했던 것과 비슷)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

# GPU가 있으면 더 자세한 정보 출력
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))


torch: 2.8.0+cu128
cuda: 12.8
cuda available: True
gpu: NVIDIA A100-SXM4-80GB
capability: (8, 0)


## 2. 프롬프트 정의

In [7]:
# ============================================================
# [7단계] AI에게 줄 "지시문(프롬프트)" 정의하기
# ============================================================
# AI에게 어떤 일을 시킬지 명확하게 알려주는 "대본"을 작성하는 단계예요.
#
# 두 가지 텍스트를 만들어요:
# 1) system_message: AI에게 "너는 어떤 역할이야"라고 알려줌
# 2) prompt: 매번 입력 데이터마다 사용할 질문 양식
# ============================================================

# 시스템 메시지: AI의 "역할/페르소나" 설정
# AI에게 "너는 패션 분류 모델이야"라고 알려주는 역할
system_message = "당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다."

# 사용자 프롬프트 템플릿: 매 입력마다 사용할 양식
# 큰따옴표 3개("""..."""): 여러 줄 문자열 (줄바꿈 포함)
#
# {name} : 나중에 실제 제품명으로 채워질 자리표시자(placeholder)
# {{ }}  : 실제 중괄호 한 개를 의미 (이스케이프)
#          파이썬 .format()에서 {} 안 글자를 변수로 인식하니까
#          중괄호 자체를 쓰려면 {{ }} 처럼 두 번 써야 함
prompt = """입력 정보:
- name: {name}
- image: [image]

위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:
1) gender
2) masterCategory
3) subCategory
4) season
5) usage
6) baseColour
7) articleType

출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:
{{
  "gender": "예시값",
  "masterCategory": "예시값",
  "subCategory": "예시값",
  "season": "예시값",
  "usage": "예시값",
  "baseColour": "예시값",
  "articleType": "예시값"
}}

# 예시
{{
  "gender": "Men",
  "masterCategory": "Accessories",
  "subCategory": "Eyewear",
  "season": "Winter",
  "usage": "Casual",
  "baseColour": "Blue",
  "articleType": "Sunglasses"
}}

# 주의
- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.
"""

# 이 프롬프트는 AI에게 다음과 같이 부탁하는 거예요:
# "옷 이미지랑 제품명 줄게. 그 옷의 7가지 속성을 JSON 형식으로 답변해줘."
# 7가지 속성: 성별, 대분류, 소분류, 시즌, 용도, 기본색상, 아이템 종류


## 3. 데이터셋 로드 및 라벨 설정

In [8]:
# ============================================================
# [8단계] 정답 라벨을 JSON 형식으로 합치는 함수 만들기
# ============================================================
# 원본 데이터는 gender, masterCategory 등이 각각 따로 칸에 있어요.
# 그런데 우리가 AI에게 가르치고 싶은 "정답"은 JSON 형식의 문자열이에요.
# (프롬프트에서 JSON으로 답해달라고 했으니까!)
#
# 그래서 흩어진 7개의 속성을 모아서 하나의 JSON 문자열로 만드는 함수예요.
# ============================================================

# def 함수이름(매개변수): 형태로 함수 정의
# example: 데이터 한 개 (딕셔너리 형태)
def combine_cols_to_label(example):
    # 7개의 속성을 딕셔너리로 묶기
    # 딕셔너리: {"키": 값, "키": 값} 형태로 데이터를 저장하는 형식
    label_dict = {
        "gender": example["gender"],                    # 성별 (Men/Women 등)
        "masterCategory": example["masterCategory"],    # 대분류 (Apparel 등)
        "subCategory": example["subCategory"],          # 소분류 (Topwear 등)
        "season": example["season"],                    # 시즌 (Fall/Winter 등)
        "usage": example["usage"],                      # 용도 (Casual 등)
        "baseColour": example["baseColour"],            # 기본색상 (Blue 등)
        "articleType": example["articleType"],          # 아이템 종류 (Tshirts 등)
    }

    # json.dumps(): 딕셔너리를 JSON 문자열로 변환
    # ensure_ascii=False: 한글이 \uXXXX 코드로 안 바뀌게 (가독성 위해)
    # 변환 결과를 example의 새 칸 'label'에 추가
    example["label"] = json.dumps(label_dict, ensure_ascii=False)

    # 라벨이 추가된 example을 돌려줌
    return example


In [9]:
# ============================================================
# [9단계] 데이터를 "대화 형식"으로 변환하는 함수 만들기
# ============================================================
# 요즘 AI들은 "ChatGPT처럼 대화하듯이" 학습돼요.
# 그래서 학습 데이터도 [시스템 메시지][사용자 질문][AI 답변] 같은
# 대화 형식으로 만들어줘야 해요.
#
# 이 함수는 데이터 한 개를 받아서 그런 "대화 시나리오"로 변환해요.
# ============================================================

def format_data(sample):
    # ─── 이미지 안전하게 RGB로 변환 ───
    # io.BytesIO(): 메모리에 가상의 파일 만들기
    # (디스크에 파일 쓰지 않고 메모리에서 처리 → 빠름)
    buffer = io.BytesIO()

    # 이미지를 PNG 형식으로 메모리(buffer)에 저장
    sample["image"].save(buffer, format="PNG")

    # buffer의 읽기 위치를 처음으로 되돌림 (파일 다시 읽기 준비)
    buffer.seek(0)

    # buffer에서 이미지를 다시 열어 RGB 모드로 변환
    # (어떤 이미지는 회색/투명 채널일 수 있어서 통일성을 위해 RGB로)
    image = Image.open(buffer).convert("RGB")

    # ─── 대화 시나리오 형식으로 반환 ───
    # 이건 ChatGPT 대화 형식과 비슷한 구조예요:
    # [시스템 안내] → [사용자 질문 + 이미지] → [AI 정답]
    return {
        "messages": [
            # ─── 1) 시스템 메시지 ───
            # AI에게 "너는 패션 분류 모델이야"라고 알려주는 부분
            {
                "role": "system",                # 역할: 시스템
                "content": [
                    {
                        "type": "text",          # 컨텐츠 타입: 텍스트
                        "text": system_message,  # 7단계의 system_message
                    }
                ],
            },

            # ─── 2) 사용자 메시지 (질문 + 이미지) ───
            {
                "role": "user",                  # 역할: 사용자
                "content": [
                    {
                        "type": "text",
                        # prompt 양식에 실제 제품명을 채워넣음
                        # productDisplayName: 데이터의 제품명 컬럼
                        "text": prompt.format(name=sample["productDisplayName"]),
                    },
                    {
                        "type": "image",         # 이미지 첨부
                        "image": image,          # 위에서 변환한 RGB 이미지
                    },
                ],
            },

            # ─── 3) AI 정답 (학습 시 모범 답안) ───
            # AI에게 "이런 질문엔 이렇게 답해야 해"라고 가르치는 부분
            {
                "role": "assistant",             # 역할: AI(어시스턴트)
                "content": [
                    {
                        "type": "text",
                        # 8단계에서 만든 JSON 문자열을 정답으로 제공
                        "text": sample["label"],
                    }
                ],
            },
        ],
    }


In [10]:
# ============================================================
# [10단계] 데이터셋 다운로드 → 라벨 추가 → 학습/테스트 분할
# ============================================================
# 이번 단계는 4가지 작업이 연속으로 일어나요:
# 1) 캐글의 패션 데이터셋 다운로드 (약 44,000개 이미지)
# 2) 8단계 함수로 JSON 정답(label) 추가
# 3) 데이터 순서 섞기 (학습 효과 향상)
# 4) 9단계 함수로 대화 형식 변환
# 5) 학습용/테스트용 데이터 분할
# ============================================================

# ─── 1) 데이터셋 다운로드 ───
# "ashraq/fashion-product-images-small":
#   - ashraq: 데이터 올린 사람
#   - 패션 제품 이미지 (작은 사이즈 버전)
# split="train": train 부분만 가져오기 (전체 데이터)
dataset = load_dataset("ashraq/fashion-product-images-small", split="train")

# ─── 2) 8단계 함수로 라벨 추가 ───
# .map(함수): 모든 데이터 항목에 함수를 적용 (각각 라벨 생성)
dataset_add_label = dataset.map(combine_cols_to_label)

# ─── 3) 데이터 순서 무작위로 섞기 ───
# 원본 순서가 어떤 패턴이 있을 수 있으니 섞어줌
# seed=4242: 매번 같은 순서로 섞이게 시드 고정
dataset_add_label = dataset_add_label.shuffle(seed=4242)

# ─── 4) 모든 데이터를 대화 형식으로 변환 ───
# 리스트 컴프리헨션: [함수(x) for x in 리스트] 형식으로 새 리스트 만듦
# 모든 데이터에 9단계 format_data 함수 적용
# (참고: 시간이 좀 걸려요. 44,000개 처리)
formatted_dataset = [format_data(row) for row in dataset_add_label]

# ─── 5) 학습용 / 테스트용 데이터 분할 ───
# train_test_split: 데이터를 두 묶음으로 나누는 도구
# test_size=0.9: 90%를 테스트용, 10%를 학습용으로 사용
#   → 일반적으론 반대(테스트 적게)지만 여기선 빠른 실험을 위해 학습 데이터를 적게
#   → 학습 4,407개 / 테스트 39,665개
# random_state=42: 매번 같게 분할되도록 시드 고정
train_dataset, test_dataset = train_test_split(
    formatted_dataset,
    test_size=0.9,
    random_state=42,
)


README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…):   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00001-of-00002-bb459e5ac5f01e(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Map:   0%|          | 0/44072 [00:00<?, ? examples/s]

In [11]:
# ============================================================
# [11단계] 데이터 잘 준비됐는지 확인하기
# ============================================================
# 분할이 잘 됐는지, 데이터 모양이 어떤지 출력해서 점검해요.
# ============================================================

# len(): 리스트의 항목 개수를 알려주는 함수
print("학습 데이터의 개수:", len(train_dataset))   # 약 4,407개
print("테스트 데이터의 개수:", len(test_dataset))  # 약 39,665개

# 학습 데이터의 첫 번째 샘플을 출력
# 출력 결과는 9단계에서 만든 "대화 형식"이 보여요
print("샘플 데이터:")
print(train_dataset[0])


학습 데이터의 개수: 4407
테스트 데이터의 개수: 39665
샘플 데이터:
{'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': '당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '입력 정보:\n- name: Mr.Men Men\'s Charcoal White T-shirt\n- image: [image]\n\n위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:\n1) gender\n2) masterCategory\n3) subCategory\n4) season\n5) usage\n6) baseColour\n7) articleType\n\n출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:\n{\n  "gender": "예시값",\n  "masterCategory": "예시값",\n  "subCategory": "예시값",\n  "season": "예시값",\n  "usage": "예시값",\n  "baseColour": "예시값",\n  "articleType": "예시값"\n}\n\n# 예시\n{\n  "gender": "Men",\n  "masterCategory": "Accessories",\n  "subCategory": "Eyewear",\n  "season": "Winter",\n  "usage": "Casual",\n  "baseColour": "Blue",\n  "articleType": "Sunglasses"\n}\n\n# 주의\n- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.\n'}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=60x80 at 0x7F427F36E300>}]}, {'role

## 4. 프로세서 및 모델 로드

In [ ]:
# ============================================================
# [12단계] 프로세서와 AI 모델 다운로드하기
# ============================================================
# 드디어 본체! Qwen3-VL 모델을 인터넷에서 다운받아 메모리에 올려요.
# (모델 크기가 약 9GB이므로 처음엔 시간이 좀 걸려요)
#
# 두 가지를 준비해요:
# 1) 프로세서: 이미지+텍스트 → AI 입력으로 변환하는 도구
# 2) 모델: 실제 AI 두뇌 (이미지 보고 텍스트 만드는 부분)
# ============================================================

# ─── 1) 프로세서 다운로드 ───
processor = AutoProcessor.from_pretrained(
    MODEL_ID,                       # 6단계에서 정한 "Qwen/Qwen3-VL-4B-Instruct"

    # 이미지의 최소/최대 픽셀 수 설정
    # 28×28: Qwen VL 모델의 기본 패치(이미지 조각) 크기
    # 256개 패치 = 256×(28×28) = 약 200,704 픽셀 (대략 448×448 이미지)
    # 너무 작은 이미지는 키우고, 너무 큰 이미지는 줄여서 효율 균형
    min_pixels=256 * 28 * 28,       # 이미지 최소 픽셀 수
    max_pixels=512 * 28 * 28,       # 이미지 최대 픽셀 수 (약 632×632)
)

# 토크나이저(글자→숫자) 패딩을 오른쪽에 추가하도록 설정
# 학습 시엔 오른쪽 패딩, 추론 시엔 왼쪽 패딩이 일반적
processor.tokenizer.padding_side = "right"

# ─── 2) 모델 다운로드 ───
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,                       # 같은 모델 이름

    # AutoModelForImageTextToText: 이미지를 보고 텍스트를 생성하는 모델
    #   ※ 일반 LLM은 AutoModelForCausalLM(텍스트 전용)을 씀.
    #     Qwen-VL은 '비전 인코더 + 언어모델'이 한 몸이라 ImageTextToText로 로드해야 함.
    #   ▎ 💡 잠깐: Qwen-VL은 왜 모델을 "통째로" 불러올까?
    #   ▎
    #   ▎ 일반 텍스트 LLM(GPT류)은 AutoModelForCausalLM으로 언어 모델 부분만 올리면 됩니다.
    #   ▎
    #   ▎ 하지만 Qwen-VL은 멀티모달 모델이에요. 구조가 이렇게 두 덩어리로 되어 있습니다:
    #   ▎
    #   ▎ [이미지] → 🖼️  비전 인코더(Vision Encoder)┐
    #   ▎                                          ├→ 언어 모델(LLM) → [텍스트 답변]
    #   ▎ [텍스트] → ────────────────────────────┘
    #   ▎
    #   ▎ 이미지를 "보려면" 비전 인코더가 반드시 필요하기 때문에, 비전 인코더 + 언어 모델 전체를 한 몸으로 불러옵니다. 그래서 로드 
    #   ▎ 클래스도 AutoModelForCausalLM(텍스트 전용)이 아니라 AutoModelForImageTextToText(이미지+텍스트)를 쓰는 거예요.
    #   ▎
    #   ▎ 이 점은 나중에 vLLM으로 서빙할 때도 똑같이 적용됩니다. 텍스트 LLM만 올리는 게 아니라 비전 인코더를 포함한 모델 전체를 vLLM에
    #   ▎ 올려야 이미지 입력을 처리할 수 있어요. (그래서 VRAM도 텍스트 전용 모델보다 더 먹습니다)

    # GPU/CPU에 자동으로 모델을 배치
    # "auto" → 빈 GPU에 우선 올리고, 부족하면 CPU에도 분산
    device_map="auto",

    # bfloat16: 메모리를 절반으로 줄이는 데이터 형식 (32비트 → 16비트)
    # bfloat16은 float16보다 안정적이고 RTX 30/40 시리즈, A100/A6000 등에서 잘 작동
    torch_dtype=torch.bfloat16,
)

# 캐시 끄기 (학습 시엔 필요 없음, gradient checkpointing과 호환되게)
model.config.use_cache = False

# 입력에 대한 gradient(기울기) 계산 활성화
# LoRA로 학습할 때 입력 임베딩에서 기울기가 잘 흐르도록 도움
# hasattr(): 해당 객체에 그 함수가 있는지 확인 (없는 모델도 있어서 안전장치)
#  임베딩 레이어 출력에 hook을 걸어서, 그 출력이 requires_grad=True가 되도록 강제합니다.
#  [입력 토큰] → [임베딩(얼림)] → ★여기서 requires_grad=True 강제★ → ... → [LoRA] → loss
#  원본 임베딩 가중치는 여전히 학습 안 하지만(얼린 상태 유지), gradient가 통과는 하도록 길을 터주는 거예요. 
#  그러면 gradient checkpointing이 backward를 정상적으로 수행할 수 있습니다.

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

## 5. 채팅 템플릿 확인

In [13]:
# ============================================================
# [13단계] 채팅 템플릿이 어떻게 적용되는지 미리 보기
# ============================================================
# AI마다 대화를 표현하는 자기만의 "포맷"이 있어요.
# 예: <|im_start|>user 내용 <|im_end|>  같은 형태.
# 우리 데이터(9단계 대화 형식)가 실제 AI 입력 양식으로 어떻게
# 변환되는지 미리 확인해보는 단계예요.
# ============================================================

# apply_chat_template:
#   "messages"(우리 데이터의 대화 부분)을 AI 모델 전용 포맷으로 변환
text = processor.apply_chat_template(
    train_dataset[0]["messages"],   # 첫 번째 데이터의 대화 부분
    tokenize=False,                 # 토큰(숫자)으로 바꾸지 말고 글자 그대로
    add_generation_prompt=False,    # "AI 답변 시작" 표시 안 붙임 (이미 답이 있음)
)

# 변환 결과 출력
# 결과를 보면 <|im_start|>system, <|im_start|>user 같은 특수 토큰들이 보여요
# - <|im_start|>...<|im_end|>: 각 발화의 시작과 끝 표시
# - <|vision_start|><|image_pad|><|vision_end|>: 이미지가 들어가는 자리 표시
print("채팅 템플릿 적용 결과:")
print(text)


채팅 템플릿 적용 결과:
<|im_start|>system
당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.<|im_end|>
<|im_start|>user
입력 정보:
- name: Mr.Men Men's Charcoal White T-shirt
- image: [image]

위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:
1) gender
2) masterCategory
3) subCategory
4) season
5) usage
6) baseColour
7) articleType

출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:
{
  "gender": "예시값",
  "masterCategory": "예시값",
  "subCategory": "예시값",
  "season": "예시값",
  "usage": "예시값",
  "baseColour": "예시값",
  "articleType": "예시값"
}

# 예시
{
  "gender": "Men",
  "masterCategory": "Accessories",
  "subCategory": "Eyewear",
  "season": "Winter",
  "usage": "Casual",
  "baseColour": "Blue",
  "articleType": "Sunglasses"
}

# 주의
- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.
<|vision_start|><|image_pad|><|vision_end|><|im_end|>
<|im_start|>assistant
{"gender": "Men", "masterCategory": "Apparel", "subCategory": "Topwear", "season": "Fall", "usage": "Casual", "baseColour": "Grey", "articleType": "Tshirts"}<|im_end|>



## 6. 로라

In [14]:
# ============================================================
# [14단계] LoRA(효율적 학습) 설정 - 핵심!
# ============================================================
# 40억 개 파라미터를 전부 학습하려면 메모리가 부족해요.
# LoRA를 쓰면 전체의 1~2%만 학습해도 비슷한 효과를 낼 수 있어요!
#
# 비유: 거대한 도서관 책을 다 새로 쓰는 게 아니라,
#       포스트잇만 붙여서 핵심 정보를 추가하는 느낌.
# ============================================================

peft_config = LoraConfig(
    # ─── LoRA의 강도 조절 ───
    lora_alpha=64,        # 학습 강도 스케일링 (LoRA가 원본에 얼마나 영향 줄지)
    lora_dropout=0.05,    # 5% 확률로 LoRA의 일부를 무작위로 꺼서 과적합 방지

    # ─── LoRA 어댑터 크기 ───
    r=64,                 # rank(랭크) = 어댑터 크기
                          # 클수록 표현력↑ 메모리↑ (보통 8~64 사이)
                          # 위의 lora_alpha와 같이 64로 설정 = 균형 잡힌 설정

    # ─── 편향(bias) 학습 안 함 ───
    bias="none",          # bias 파라미터는 학습 안 함 (계산 절약)

    # ─── 어떤 부품에 LoRA 어댑터 끼울지 지정 ───
    # 트랜스포머 모델의 핵심 7개 부품에 LoRA 적용
    target_modules=[
        "q_proj",         # Query 프로젝션 (어텐션의 "무엇을 찾을지")
        "k_proj",         # Key 프로젝션 (어텐션의 "무엇이 있는지")
        "v_proj",         # Value 프로젝션 (어텐션의 "내용물")
        "o_proj",         # Output 프로젝션 (어텐션 결과 가공)
        "gate_proj",      # MLP의 게이트 부분 (정보 필터링)
        "up_proj",        # MLP의 확장 부분 (차원 늘리기)
        "down_proj",      # MLP의 축소 부분 (차원 줄이기)
    ],

    # ─── 작업 유형 ───
    # CAUSAL_LM: 인과적 언어 모델 (다음 단어 예측 방식)
    # GPT 계열 모델들이 사용하는 학습 방식
    task_type="CAUSAL_LM",
)


## 7. Collate_fn 설정

In [15]:
# ============================================================
# [15단계] 특수 토큰 ID 찾는 함수 만들기 (Collator 보조 함수)
# ============================================================
# Qwen VL 모델은 이미지를 표현하기 위해 특수 토큰을 사용해요:
# - <|vision_start|>: "이제부터 이미지 정보야"
# - <|image_pad|>: 이미지 한 조각의 자리 표시
# - <|vision_end|>: "이미지 정보 끝"
# - <|video_pad|>: 비디오의 자리 표시
#
# 이 특수 토큰들은 학습에서 점수 계산에 포함하면 안 돼요.
# (AI가 글자 만드는 학습을 하는 거지, 이미지 토큰을 외우는 게 아니니까)
#
# 그래서 먼저 이 특수 토큰들의 ID(숫자)를 찾는 함수를 만들어요.
# ============================================================

def get_special_token_ids(tokenizer):
    # 찾고 싶은 특수 토큰 목록
    special_tokens = [
        "<|vision_start|>",
        "<|vision_end|>",
        "<|image_pad|>",
        "<|video_pad|>",
    ]

    # 빈 리스트 만들기 (찾은 ID들을 여기에 모음)
    token_ids = []

    # 각 토큰에 대해 반복
    for token in special_tokens:
        # 토큰 문자를 ID(숫자)로 변환
        token_id = tokenizer.convert_tokens_to_ids(token)

        # ID가 없으면(None이면) 건너뛰기
        # continue: 현재 반복 건너뛰고 다음 반복으로
        if token_id is None:
            continue

        # unk_token(unknown 토큰)과 같은 ID면 건너뛰기
        # → "이 토큰을 모르겠다"는 신호이므로 무시
        if tokenizer.unk_token_id is not None and token_id == tokenizer.unk_token_id:
            continue

        # 통과한 ID는 목록에 추가
        token_ids.append(token_id)

    # set(): 중복 제거 → list(): 다시 리스트로
    # (같은 ID가 두 번 들어가지 않게)
    return list(set(token_ids))


In [16]:
# ============================================================
# [16단계] 시퀀스 안에서 부분 시퀀스 찾는 함수 (Collator 보조)
# ============================================================
# 토큰 숫자 배열 안에서 "어디부터가 AI 답변 시작인지" 찾기 위한 도구예요.
# 마치 책 본문에서 "특정 단어 위치"를 찾는 것과 같아요.
#
# 예) 큰 배열 [1, 2, 3, 4, 5, 6] 에서 [3, 4]를 찾으면 → 위치 2 반환
# 못 찾으면 -1 반환
# ============================================================

def find_subsequence(sequence, subsequence):
    # 찾을 부분 시퀀스가 비어있으면 의미가 없으니 -1 반환
    if len(subsequence) == 0:
        return -1

    # 마지막으로 매치된 위치 저장 (못 찾으면 -1로 유지)
    last_match = -1

    # range(시작, 끝): 시작부터 끝-1까지의 숫자 생성
    # len(sequence) - len(subsequence) + 1:
    #   부분 시퀀스가 들어갈 수 있는 마지막 시작 위치 + 1
    for i in range(len(sequence) - len(subsequence) + 1):
        # 슬라이싱: sequence[i:i+N] = i부터 i+N-1까지 N개 추출
        # 그 부분이 subsequence와 같으면 매치!
        if sequence[i:i + len(subsequence)] == subsequence:
            last_match = i   # 위치 저장 (덮어쓰기 → 가장 마지막 위치만 남음)

    return last_match


In [17]:
# ============================================================
# [17단계] "AI 답변 외 부분"을 학습에서 제외하는 함수
# ============================================================
# 이 함수가 왜 필요한가요?
#
# 학습 데이터에는 [시스템 메시지][사용자 질문][AI 답변]이 다 들어있어요.
# 하지만 우리가 AI에게 가르치고 싶은 건 "AI 답변 부분"이에요.
# (시스템 메시지나 사용자 질문은 AI가 만들 게 아니라 입력이니까)
#
# 그래서 정답(labels)에서 [시스템+사용자 질문] 부분을 -100으로 가려서
# "이 부분은 손실 계산에서 빼"라고 표시해줘요.
#
# 학생이 시험 볼 때 보기에서 답만 채점하는 것과 비슷해요.
# ============================================================

def mask_non_assistant_tokens(labels, input_ids, tokenizer):
    # "어시스턴트(AI) 답변 시작" 표시 문자열
    # Qwen 모델 포맷: <|im_start|>assistant\n 다음부터 AI 답변이 시작됨
    assistant_prefix = "<|im_start|>assistant\n"

    # 위 문자열을 토큰 ID 배열로 변환
    # add_special_tokens=False: 추가 특수 토큰 안 붙이게
    assistant_prefix_ids = tokenizer.encode(
        assistant_prefix,
        add_special_tokens=False,
    )

    # 배치 안의 각 데이터(행)에 대해 처리
    # input_ids.shape[0]: 배치 안 데이터 개수
    for row_idx in range(input_ids.shape[0]):
        # 텐서를 일반 리스트로 변환 (find_subsequence 함수가 리스트를 받아서)
        row_input_ids = input_ids[row_idx].tolist()

        # 16단계 함수로 "AI 답변 시작" 위치 찾기
        start_idx = find_subsequence(
            row_input_ids,
            assistant_prefix_ids,
        )

        # 못 찾았다면 → 이 데이터는 학습 안 함 (전체를 -100으로 가림)
        if start_idx == -1:
            labels[row_idx, :] = -100   # 모든 토큰 가리기
            continue

        # 답변이 시작되는 진짜 위치 = 시작 표시 + 그 표시의 길이
        answer_start_idx = start_idx + len(assistant_prefix_ids)

        # 답변 시작 이전까지 모두 -100으로 가림
        # [row_idx, :answer_start_idx]: 그 행의 시작부터 answer_start_idx 직전까지
        labels[row_idx, :answer_start_idx] = -100

    return labels


In [18]:
# ============================================================
# [18단계] 메인 콜레이터 함수 만들기 (가장 복잡한 부분!)
# ============================================================
# 콜레이터: 데이터 여러 개를 한 묶음(batch)으로 만들어주는 도구.
# AI 학습 시 매번 호출돼서 데이터를 적절한 형태로 가공해줘요.
#
# 이 함수가 하는 일:
# 1) 텍스트 처리: 대화를 채팅 템플릿 적용 후 토큰화
# 2) 이미지 처리: 데이터에서 이미지/비디오 추출
# 3) 전체 처리: 프로세서로 한꺼번에 처리 (텍스트+이미지+패딩)
# 4) 라벨 설정: 학습에서 빼야 할 부분을 -100으로 가림
# ============================================================

def collate_fn(examples):
    # ─── 1) 대화 → 텍스트 변환 ───
    # 모든 예시의 messages를 채팅 템플릿 적용해서 문자열로 변환
    # (리스트 컴프리헨션으로 한꺼번에)
    texts = [
        processor.apply_chat_template(
            example["messages"],
            tokenize=False,                 # 글자 그대로 유지
            add_generation_prompt=False,    # 학습 시엔 답변 시작 표시 안 붙임
        )
        for example in examples
    ]

    # ─── 2) 이미지/비디오 추출 ───
    image_inputs = []   # 이미지 모음
    video_inputs = []   # 비디오 모음 (이번 작업엔 없지만 안전하게)

    for example in examples:
        # process_vision_info: Qwen VL 보조 도구로 messages에서 자동 추출
        image_input, video_input = process_vision_info(example["messages"])

        # 이미지가 있으면 모음에 추가
        # .extend(): 리스트의 항목들을 펼쳐서 추가 (vs .append: 통째로 추가)
        if image_input is not None:
            image_inputs.extend(image_input)

        if video_input is not None and len(video_input) > 0:
            video_inputs.extend(video_input)

    # ─── 3) 프로세서에 전달할 인자 준비 ───
    # 딕셔너리에 필수 인자들 담아놓기
    processor_kwargs = {
        "text": texts,                  # 텍스트 목록
        "return_tensors": "pt",         # PyTorch 텐서 형식
        "padding": True,                # 짧은 데이터 뒤에 패딩 추가
    }

    # 이미지가 있으면 인자에 추가
    if len(image_inputs) > 0:
        processor_kwargs["images"] = image_inputs

    # 비디오가 있으면 인자에 추가
    if len(video_inputs) > 0:
        processor_kwargs["videos"] = video_inputs

    # 프로세서 호출 (텍스트+이미지를 한꺼번에 AI 입력으로 변환)
    # **딕셔너리: 딕셔너리를 함수의 키워드 인자로 펼쳐서 전달
    batch = processor(**processor_kwargs)

    # ─── 4) 라벨(정답) 만들기 ───
    # 일단 input_ids(입력 토큰)를 복사해서 라벨로 사용
    # .clone(): 텐서를 복사 (원본에 영향 안 주게)
    labels = batch["input_ids"].clone()

    # 패딩 토큰 위치를 -100으로 가림 (손실 계산에서 제외)
    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100

    # 특수 토큰 위치도 모두 -100으로 가림
    # (이미지 자리 표시 같은 건 학습 대상이 아님)
    for special_token_id in get_special_token_ids(processor.tokenizer):
        labels[labels == special_token_id] = -100

    # 17단계 함수로 "AI 답변 외 부분" 가리기
    labels = mask_non_assistant_tokens(
        labels=labels,
        input_ids=batch["input_ids"],
        tokenizer=processor.tokenizer,
    )

    # 완성된 라벨을 배치에 추가
    batch["labels"] = labels

    # 최종 배치 반환 (이게 AI 학습 한 입 분량)
    return batch


## 8. Collate 함수 테스트

In [19]:
# ============================================================
# [19단계] 콜레이터가 잘 작동하는지 테스트하기
# ============================================================
# 본격 학습 전에 18단계 콜레이터가 정상 작동하는지 확인해요.
# 데이터 1개를 콜레이터에 넣어보고 결과를 들여다봐요.
# ============================================================

# 단일 예시 확인
example = train_dataset[0]  # 데이터셋의 첫 번째 아이템

# 원본 데이터 출력 (어떻게 생겼는지)
print("단일 예시 데이터:")
print(example)

# collate_fn 테스트 (예시 1개만 리스트로 묶어 전달, 즉 배치 크기 1)
batch = collate_fn([example])

# ─── 결과 확인 ───
print("\n처리된 배치 데이터:")

# 입력 ID(토큰 숫자 배열)의 크기
# [1, N] → 1개 데이터, N개 토큰
print("입력 ID 형태:", batch["input_ids"].shape)

# 어텐션 마스크: 어느 토큰이 진짜인지(1), 패딩인지(0) 표시
print("어텐션 마스크 형태:", batch["attention_mask"].shape)

# 이미지 픽셀 값 (있을 때만 출력)
# "키" in 딕셔너리: 그 키가 딕셔너리에 있는지 확인
if "pixel_values" in batch:
    print("이미지 픽셀 형태:", batch["pixel_values"].shape)

# 이미지 grid 정보: 이미지를 몇 조각으로 나눴는지 (시간, 높이, 너비)
if "image_grid_thw" in batch:
    print("이미지 grid 형태:", batch["image_grid_thw"].shape)
    print("이미지 grid 값:")
    print(batch["image_grid_thw"])

# 비디오 grid 정보 (있을 때만)
if "video_grid_thw" in batch:
    print("비디오 grid 형태:", batch["video_grid_thw"].shape)
    print("비디오 grid 값:")
    print(batch["video_grid_thw"])


단일 예시 데이터:
{'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': '당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': '입력 정보:\n- name: Mr.Men Men\'s Charcoal White T-shirt\n- image: [image]\n\n위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:\n1) gender\n2) masterCategory\n3) subCategory\n4) season\n5) usage\n6) baseColour\n7) articleType\n\n출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:\n{\n  "gender": "예시값",\n  "masterCategory": "예시값",\n  "subCategory": "예시값",\n  "season": "예시값",\n  "usage": "예시값",\n  "baseColour": "예시값",\n  "articleType": "예시값"\n}\n\n# 예시\n{\n  "gender": "Men",\n  "masterCategory": "Accessories",\n  "subCategory": "Eyewear",\n  "season": "Winter",\n  "usage": "Casual",\n  "baseColour": "Blue",\n  "articleType": "Sunglasses"\n}\n\n# 주의\n- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.\n'}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=60x80 at 0x7F427F36E300>}]}, {'role': 'assistant', 'content': [{'typ

In [20]:
# ============================================================
# [20단계] 라벨이 어떻게 구성됐는지 자세히 보기
# ============================================================
# 라벨(정답) 토큰들이 어떻게 가공됐는지 들여다봐요.
# 마스킹(-100으로 가리기)이 잘 적용됐는지 확인하는 단계예요.
# ============================================================

# 라벨 텐서의 모양 확인 (입력과 같은 크기여야 함)
print("레이블 형태:", batch["labels"].shape)

# 입력 ID 전체 확인
# 0번 데이터(우리는 1개만 있음)의 토큰 숫자들
print("\n입력에 대한 정수 인코딩 결과:")
print(batch["input_ids"][0])

# 레이블 전체 확인
# 출력에 -100이 많이 보일 거예요
# -100은 "여긴 학습 안 함"이라는 표시
# (시스템 메시지, 사용자 질문, 특수 토큰 부분 = 모두 -100)
print("\n레이블에 대한 정수 인코딩 결과:")
print(batch["labels"][0])


레이블 형태: torch.Size([1, 575])

입력에 대한 정수 인코딩 결과:
tensor([151644,   8948,    198,  64795,  82528,  33704,  90667,  21329,  80573,
        138017,  79632,   3153,      8,  42039, 126558,  45104,    101,  92031,
            14, 141274,  32077,  60039,  18411,  57835, 126605,  42905, 128618,
         97929,  54070, 142713,  78952,     13, 151645,    198, 151644,    872,
           198,  43866,  28754,  60039,    510,     12,    829,     25,   4392,
          1321,    268,  11012,    594,   4864,  40465,   5807,    350,  33668,
           198,     12,   2168,     25,    508,   1805,   2533,  80901,  60039,
         18411,  81718, 144059,  42039,     11, 136646,    220,     22,  19969,
         21329,   1376,  19391, 128605,  93668,   4718, 141966,  17380,  57835,
        126605,  33883,  55673,  50302,    510,     16,      8,   9825,    198,
            17,      8,   7341,   6746,    198,     18,      8,   1186,   6746,
           198,     19,      8,   3200,    198,     20,      8,  10431, 

In [21]:
# ============================================================
# [21단계] 토큰을 다시 글자로 되돌려서 사람이 확인하기
# ============================================================
# 숫자만 봐서는 잘 안 보이니까, 글자로 다시 변환해서
# "AI가 보는 내용"과 "실제 학습 대상"을 사람 눈으로 확인해요.
# ============================================================

# ─── 1) 전체 입력을 글자로 변환 ───
# decode(): 토큰 숫자 → 글자 변환 (encode의 반대)
decoded_text = processor.tokenizer.decode(batch["input_ids"][0])

# 출력 결과를 보면 <|im_start|>, <|image_pad|> 같은 특수 토큰들이 보여요
# 이미지가 들어가는 자리에 <|image_pad|>가 수백 개 반복돼 있을 거예요
# (이미지를 잘게 쪼개 각 조각마다 하나씩)
print("\n디코딩된 텍스트:")
print(decoded_text)

# ─── 2) 실제 학습 대상 토큰만 확인 ───
# (-100이 아닌 부분 = 실제로 점수 계산되는 부분)
# .sum(): True/False 값을 합산 (True=1, False=0)
# != -100: -100이 아닌 위치를 True로
# .item(): 1개짜리 텐서를 일반 숫자로
valid_label_count = (batch["labels"][0] != -100).sum().item()

print("\nloss 계산 대상 토큰 수:")
print(valid_label_count)   # 약 53개 (JSON 답변 길이만큼)

# ─── 3) -100을 제외한 라벨만 글자로 변환 ───
label_ids = batch["labels"][0]

# 마스킹: label_ids != -100 → True/False 배열
# 그 배열을 인덱스로 쓰면 -100이 아닌 것만 골라짐
label_ids_for_decode = label_ids[label_ids != -100]

# 추출된 토큰을 글자로 변환
decoded_labels = processor.tokenizer.decode(label_ids_for_decode)

# 출력: JSON 형식의 답변만 보여야 정상!
# 시스템 메시지나 사용자 질문은 안 보여야 함
print("\nloss 계산 대상 labels 디코딩 결과:")
print(decoded_labels)



디코딩된 텍스트:
<|im_start|>system
당신은 이미지와 제품명(name)으로부터 패션/스타일 정보를 추론하는 분류 모델입니다.<|im_end|>
<|im_start|>user
입력 정보:
- name: Mr.Men Men's Charcoal White T-shirt
- image: [image]

위 정보를 바탕으로, 아래 7가지 key에 대한 값을 JSON 형태로 추론해 주세요:
1) gender
2) masterCategory
3) subCategory
4) season
5) usage
6) baseColour
7) articleType

출력 시 **아래 JSON 예시 형태**를 반드시 지키세요:
{
  "gender": "예시값",
  "masterCategory": "예시값",
  "subCategory": "예시값",
  "season": "예시값",
  "usage": "예시값",
  "baseColour": "예시값",
  "articleType": "예시값"
}

# 예시
{
  "gender": "Men",
  "masterCategory": "Accessories",
  "subCategory": "Eyewear",
  "season": "Winter",
  "usage": "Casual",
  "baseColour": "Blue",
  "articleType": "Sunglasses"
}

# 주의
- 7개 항목 이외의 정보(텍스트, 문장 등)는 절대 포함하지 마세요.
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_p

## 9. 학습 설정

In [22]:
# ============================================================
# [22단계] 학습 옵션 설정 (운동 계획표 짜기!)
# ============================================================
# SFTConfig: SFT(Supervised Fine-Tuning, 지도 미세조정) 설정
# AI를 어떻게 학습시킬지 세부 옵션을 정해요.
# ============================================================

args = SFTConfig(
    # ─── 저장 ───
    output_dir=OUTPUT_DIR,                 # 6단계의 저장 폴더

    # ─── 학습 횟수 ───
    # 1 epoch = 전체 데이터를 한 번 다 본 것
    num_train_epochs=2,                    # 데이터 전체를 2번 반복 학습

    # ─── 배치 크기 ───
    # GPU 메모리에 따라 조절 (작을수록 안정적, 클수록 빠름)
    per_device_train_batch_size=2,         # GPU당 한 번에 2개씩

    # gradient_accumulation_steps=8:
    # 2개씩 8번 모아서 실질적 배치 = 16개로 학습
    # → 메모리 부족할 때 효과적인 트릭
    gradient_accumulation_steps=8,

    # ─── 메모리 절약 기법 ───
    # 학습 중 중간 계산값을 저장하지 않고 다시 계산하는 방식
    # → 메모리 절약, 대신 속도 약간 느려짐 (트레이드오프)
    gradient_checkpointing=True,

    # 옵티마이저: AI의 학습 방식을 조정하는 알고리즘
    # adamw_torch_fused: 빠르고 안정적인 옵션 (CUDA 가속 적용)
    optim="adamw_torch_fused",

    # ─── 로그 ───
    logging_steps=10,                      # 10 step마다 진행 상황 출력

    # ─── 모델 저장 전략 ───
    save_strategy="steps",                 # step 단위로 저장
    save_steps=50,                         # 50 step마다 체크포인트 저장
    save_total_limit=3,                    # 최대 3개까지만 (오래된 건 삭제)

    # ─── 수치 정밀도 ───
    bf16=True,                             # bfloat16 사용 (메모리/속도 효율)
    fp16=False,                            # fp16은 안 씀 (둘 중 하나만)

    # ─── 학습률 관련 ───
    learning_rate=1e-4,                    # 학습률 = 0.0001
                                           # AI가 한 번에 얼마나 크게 변할지

    max_grad_norm=0.3,                     # 기울기 최대 크기 제한
                                           # → 학습 폭주(loss 튀는 현상) 방지

    warmup_ratio=0.03,                     # 처음 3%는 천천히 워밍업
    lr_scheduler_type="constant",          # 학습률 일정하게 유지

    # ─── 기타 ───
    push_to_hub=False,                     # 학습 후 허깅페이스에 업로드 안 함
    remove_unused_columns=False,           # 데이터의 안 쓰는 컬럼 제거 안 함

    # 데이터셋 자동 준비 건너뛰기
    # (우리가 직접 collate_fn에서 처리하니까)
    dataset_kwargs={
        "skip_prepare_dataset": True,
    },

    # 실험 추적 도구 사용 안 함
    report_to=None,
)


## 10. Trainer 생성

In [23]:
# ============================================================
# [23단계] 드디어 학습 시작! (마지막 단계)
# ============================================================
# 지금까지 22단계 동안 준비한 모든 것을 모아 학습을 시작해요.
#
# SFTTrainer가 알아서 다음 일들을 진행해줘요:
# 1) 데이터를 collate_fn으로 묶기
# 2) 모델에 LoRA 어댑터 부착
# 3) AI에게 데이터를 먹이고 정답과 비교
# 4) AI의 LoRA 부품만 살짝씩 수정
# 5) 일정 step마다 체크포인트 저장
#
# 학습 결과 예시 (실행 후 출력):
#   - global_step=552: 552번 학습 완료
#   - training_loss=0.036: 손실값 0.036 (낮음 = 잘 배움)
#   - train_runtime=4417초 (약 73분 소요)
#
# [주의] 학습 시간: GPU 1개 기준 약 1~2시간
# ============================================================

# 트레이너(학습 코치) 생성
trainer = SFTTrainer(
    model=model,                       # 12단계의 Qwen3-VL 모델
    args=args,                         # 22단계의 학습 옵션
    train_dataset=train_dataset,       # 10단계의 학습 데이터 (4,407개)
    data_collator=collate_fn,          # 18단계의 콜레이터
    peft_config=peft_config,           # 14단계의 LoRA 설정
    processing_class=processor,        # 12단계의 프로세서
                                       # (최신 trl 버전에서 tokenizer 대신 사용)
)

# 실제 학습 실행!
# 이 한 줄이 실행되면 자동으로:
#   - 학습 데이터를 차례차례 AI에 먹임
#   - AI의 답과 정답을 비교해 손실(loss) 계산
#   - 그 손실을 줄이는 방향으로 LoRA 파라미터 업데이트
#   - 진행률, 손실 값 등을 화면에 출력
trainer.train()


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.304200
20,0.079600
30,0.057500
40,0.049700
50,0.045800
60,0.034700
70,0.035400
80,0.043100
90,0.044100
100,0.033200


TrainOutput(global_step=552, training_loss=0.03674887494602497, metrics={'train_runtime': 2860.0149, 'train_samples_per_second': 3.082, 'train_steps_per_second': 0.193, 'total_flos': 1.2705585830232576e+17, 'train_loss': 0.03674887494602497})